# Data bilan tanishuv 

In [187]:
import pandas as pd 
df=pd.read_csv('imdb_top_1000.csv')

In [188]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Poster_Link    1000 non-null   str    
 1   Series_Title   1000 non-null   str    
 2   Released_Year  1000 non-null   str    
 3   Certificate    899 non-null    str    
 4   Runtime        1000 non-null   str    
 5   Genre          1000 non-null   str    
 6   IMDB_Rating    1000 non-null   float64
 7   Overview       1000 non-null   str    
 8   Meta_score     843 non-null    float64
 9   Director       1000 non-null   str    
 10  Star1          1000 non-null   str    
 11  Star2          1000 non-null   str    
 12  Star3          1000 non-null   str    
 13  Star4          1000 non-null   str    
 14  No_of_Votes    1000 non-null   int64  
 15  Gross          831 non-null    str    
dtypes: float64(2), int64(1), str(13)
memory usage: 125.1 KB


In [189]:
df.isnull().sum()

Poster_Link        0
Series_Title       0
Released_Year      0
Certificate      101
Runtime            0
Genre              0
IMDB_Rating        0
Overview           0
Meta_score       157
Director           0
Star1              0
Star2              0
Star3              0
Star4              0
No_of_Votes        0
Gross            169
dtype: int64

In [190]:
df.drop(['Poster_Link', 'Series_Title', 'Overview'], axis=1, inplace=True)

In [191]:
df.sample(5)

,Released_Year,Certificate,Runtime,Genre,IMDB_Rating,Meta_score,Director,Star1,Star2,Star3,Star4,No_of_Votes,Gross
843,1979,U,100 min,"Animation, Adventure, Family",7.7,71.0,Hayao Miyazaki,Yasuo Yamada,Eiko Masuyama,Kiyoshi Kobayashi,Makio Inoue,27014,NaN
276,1980,UA,124 min,"Biography, Drama",8.1,78.0,David Lynch,Anthony Hopkins,John Hurt,Anne Bancroft,John Gielgud,220078,NaN
75,1979,R,117 min,"Horror, Sci-Fi",8.4,89.0,Ridley Scott,Sigourney Weaver,Tom Skerritt,John Hurt,Veronica Cartwright,787806,"78,900,000"
65,2007,U,165 min,"Drama, Family",8.4,NaN,Aamir Khan,Amole Gupte,Darsheel Safary,Aamir Khan,Tisca Chopra,168895,"1,223,869"
789,2002,R,115 min,"Comedy, Drama",7.7,83.0,Spike Jonze,Nicolas Cage,Meryl Streep,Chris Cooper,Tilda Swinton,178565,"22,245,861"


In [192]:
df.isnull().sum()

Released_Year      0
Certificate      101
Runtime            0
Genre              0
IMDB_Rating        0
Meta_score       157
Director           0
Star1              0
Star2              0
Star3              0
Star4              0
No_of_Votes        0
Gross            169
dtype: int64

In [193]:
df['Gross'] = df['Gross'].str.replace(',', '').astype(float)

In [194]:
df['Meta_score'] = df['Meta_score'].fillna(df['Meta_score'].mean())
df['Gross'] = df['Gross'].fillna(df['Gross'].mean())
df['Certificate'] = df['Certificate'].fillna(df['Certificate'].mode()[0])
df['Runtime'] = df['Runtime'].str.replace(' min', '').astype(int)

In [195]:
df.isnull().sum()

Released_Year    0
Certificate      0
Runtime          0
Genre            0
IMDB_Rating      0
Meta_score       0
Director         0
Star1            0
Star2            0
Star3            0
Star4            0
No_of_Votes      0
Gross            0
dtype: int64

In [196]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

class Datapreprocessing:
    def __init__(self, df):
        self.df = df.copy()
    
    # Handle missing values
    def tozala(self):
        for col in self.df.columns:
            if self.df[col].isnull().any():
                if self.df[col].dtype == 'str':
                    self.df[col].fillna(self.df[col].mode()[0], inplace=True)
                else:
                    self.df[col].fillna(self.df[col].mean(), inplace=True)
        return self
    
    # Encode categorical variables
    def encodla(self):
        encoder = LabelEncoder()
        for col in self.df.columns:
            if self.df[col].dtype == 'str':
                if self.df[col].nunique() <= 5:  # few categories → OneHot
                    dummies = pd.get_dummies(self.df[col], prefix=col, dtype=int)
                    self.df = pd.concat([self.df.drop(columns=[col]), dummies], axis=1)
                else:  # many categories → Label Encoding
                    self.df[col] = encoder.fit_transform(self.df[col])
        return self
    
    # Scale numeric features
    def scale_qil(self):
        scaler = MinMaxScaler()
        num_cols = self.df.select_dtypes(include=['int64', 'float64']).columns.drop('xG', errors='ignore')
        self.df[num_cols] = scaler.fit_transform(self.df[num_cols])
        return self
    
    # Log transformation for skewed features
    def log_transform(self, threshold=0.5):
        skewness = self.df.skew()
        features_log = skewness[(skewness >= threshold)].index.tolist()
        
        for col in features_log:
            if (self.df[col] > 0).all():  # only positive values
                self.df[col] = np.log1p(self.df[col])
        return self


In [197]:
processor = Datapreprocessing(df)

df_ready = (
    processor
    .tozala()
    .encodla()
    .scale_qil()
    .log_transform()
    .df
)

print(df_ready.head())

   Released_Year  Certificate   Runtime     Genre  IMDB_Rating  Meta_score  \
0       0.727273     0.066667  0.351449  0.681592     1.000000    0.722222   
1       0.505051     0.066667  0.471014  0.606965     0.941176    1.000000   
2       0.868687     0.933333  0.387681  0.109453     0.823529    0.777778   
3       0.525253     0.066667  0.568841  0.606965     0.823529    0.861111   
4       0.353535     0.800000  0.184783  0.606965     0.823529    0.944444   

   Director     Star1     Star2     Star3     Star4  No_of_Votes     Gross  
0  0.257770  0.908953  0.676190  0.100000  0.972281     1.000000  0.030257  
1  0.250457  0.632777  0.010714  0.377528  0.206823     0.688207  0.144092  
2  0.151737  0.194234  0.336905  0.001124  0.660981     0.982797  0.571025  
3  0.250457  0.013657  0.782143  0.791011  0.206823     0.476641  0.061173  
4  0.833638  0.382398  0.552381  0.601124  0.448827     0.286778  0.004653  


In [198]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Released_Year  1000 non-null   str    
 1   Certificate    1000 non-null   str    
 2   Runtime        1000 non-null   int64  
 3   Genre          1000 non-null   str    
 4   IMDB_Rating    1000 non-null   float64
 5   Meta_score     1000 non-null   float64
 6   Director       1000 non-null   str    
 7   Star1          1000 non-null   str    
 8   Star2          1000 non-null   str    
 9   Star3          1000 non-null   str    
 10  Star4          1000 non-null   str    
 11  No_of_Votes    1000 non-null   int64  
 12  Gross          1000 non-null   float64
dtypes: float64(3), int64(2), str(8)
memory usage: 101.7 KB


In [199]:
df=df_ready

In [200]:
df.isnull().sum()

Released_Year    0
Certificate      0
Runtime          0
Genre            0
IMDB_Rating      0
Meta_score       0
Director         0
Star1            0
Star2            0
Star3            0
Star4            0
No_of_Votes      0
Gross            0
dtype: int64

# Filter Methot

In [201]:
import numpy as np

corr_matrix = df.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.8)]
df_filtered = df.drop(columns=to_drop)
print("\nOriginal shape:", df.shape)
print("Shape after dropping correlated features:", df_filtered.shape)


Original shape: (1000, 13)
Shape after dropping correlated features: (1000, 13)


In [202]:
df_filtered.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Released_Year  1000 non-null   float64
 1   Certificate    1000 non-null   float64
 2   Runtime        1000 non-null   float64
 3   Genre          1000 non-null   float64
 4   IMDB_Rating    1000 non-null   float64
 5   Meta_score     1000 non-null   float64
 6   Director       1000 non-null   float64
 7   Star1          1000 non-null   float64
 8   Star2          1000 non-null   float64
 9   Star3          1000 non-null   float64
 10  Star4          1000 non-null   float64
 11  No_of_Votes    1000 non-null   float64
 12  Gross          1000 non-null   float64
dtypes: float64(13)
memory usage: 101.7 KB


In [203]:
import seaborn as sns
import matplotlib.pyplot as plt
corr_matrix = df.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.8)]

In [204]:
corr_matrix = df.corr().abs()
print(corr_matrix)

               Released_Year  Certificate   Runtime     Genre  IMDB_Rating  \
Released_Year       1.000000     0.122704  0.165889  0.178312     0.132314   
Certificate         0.122704     1.000000  0.016269  0.120414     0.024708   
Runtime             0.165889     0.016269  1.000000  0.062285     0.243096   
Genre               0.178312     0.120414  0.062285  1.000000     0.028939   
IMDB_Rating         0.132314     0.024708  0.243096  0.028939     1.000000   
Meta_score          0.289446     0.032409  0.027794  0.138065     0.253903   
Director            0.098546     0.036088  0.016764  0.051014     0.011944   
Star1               0.014188     0.040511  0.001729  0.051163     0.019455   
Star2               0.055974     0.004142  0.056766  0.017573     0.029169   
Star3               0.000232     0.026168  0.011570  0.006990     0.016579   
Star4               0.008729     0.034269  0.031742  0.023571     0.023168   
No_of_Votes         0.241624     0.054933  0.173264  0.177041   

In [205]:
high_corr_pairs = []
for col in upper.columns:
    for row in upper.index:
        if upper.loc[row, col] is not np.nan and upper.loc[row, col] > 0.5:
            high_corr_pairs.append([row, col, round(upper.loc[row, col], 2)])

corr_table = pd.DataFrame(high_corr_pairs, columns=['Feature 1', 'Feature 2', 'Correlation'])
print("Highly correlated feature pairs (correlation > 0.5):")
print(corr_table)

Highly correlated feature pairs (correlation > 0.5):
     Feature 1 Feature 2  Correlation
0  No_of_Votes     Gross         0.55


In [206]:
df_filtered = df.drop(columns=to_drop)
print("\nOriginal shape:", df.shape)
print("Shape after dropping correlated features:", df_filtered.shape)


Original shape: (1000, 13)
Shape after dropping correlated features: (1000, 13)


# Vizual ko'rinishlar

In [207]:
import numpy as np

corr_matrix = df.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.8)]
df_filtered = df.drop(columns=to_drop)
print("\nOriginal shape:", df.shape)
print("Shape after dropping correlated features:", df_filtered.shape)


Original shape: (1000, 13)
Shape after dropping correlated features: (1000, 13)


In [208]:
import numpy as np

corr_matrix = df.corr().abs()
corr_matrix

,Released_Year,Certificate,Runtime,Genre,IMDB_Rating,Meta_score,Director,Star1,Star2,Star3,Star4,No_of_Votes,Gross
Released_Year,1.000000,0.122704,0.165889,0.178312,0.132314,0.289446,0.098546,0.014188,0.055974,0.000232,0.008729,0.241624,0.194017
Certificate,0.122704,1.000000,0.016269,0.120414,0.024708,0.032409,0.036088,0.040511,0.004142,0.026168,0.034269,0.054933,0.170909
Runtime,0.165889,0.016269,1.000000,0.062285,0.243096,0.027794,0.016764,0.001729,0.056766,0.011570,0.031742,0.173264,0.124626
Genre,0.178312,0.120414,0.062285,1.000000,0.028939,0.138065,0.051014,0.051163,0.017573,0.006990,0.023571,0.177041,0.299594
IMDB_Rating,0.132314,0.024708,0.243096,0.028939,1.000000,0.253903,0.011944,0.019455,0.029169,0.016579,0.023168,0.494979,0.089881
Meta_score,0.289446,0.032409,0.027794,0.138065,0.253903,1.000000,0.062344,0.023295,0.003278,0.028885,0.004487,0.017739,0.030905
Director,0.098546,0.036088,0.016764,0.051014,0.011944,0.062344,1.000000,0.039417,0.040011,0.056158,0.035279,0.015364,0.048258
Star1,0.014188,0.040511,0.001729,0.051163,0.019455,0.023295,0.039417,1.000000,0.020884,0.006638,0.006605,0.025073,0.011281
Star2,0.055974,0.004142,0.056766,0.017573,0.029169,0.003278,0.040011,0.020884,1.000000,0.012531,0.039116,0.035432,0.001477
Star3,0.000232,0.026168,0.011570,0.006990,0.016579,0.028885,0.056158,0.006638,0.012531,1.000000,0.026552,0.047634,0.005570


In [209]:
df_filtered.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Released_Year  1000 non-null   float64
 1   Certificate    1000 non-null   float64
 2   Runtime        1000 non-null   float64
 3   Genre          1000 non-null   float64
 4   IMDB_Rating    1000 non-null   float64
 5   Meta_score     1000 non-null   float64
 6   Director       1000 non-null   float64
 7   Star1          1000 non-null   float64
 8   Star2          1000 non-null   float64
 9   Star3          1000 non-null   float64
 10  Star4          1000 non-null   float64
 11  No_of_Votes    1000 non-null   float64
 12  Gross          1000 non-null   float64
dtypes: float64(13)
memory usage: 101.7 KB


In [210]:
import seaborn as sns
import matplotlib.pyplot as plt
corr_matrix = df.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.8)]

In [211]:
corr_matrix = df.corr().abs()
print(corr_matrix)

               Released_Year  Certificate   Runtime     Genre  IMDB_Rating  \
Released_Year       1.000000     0.122704  0.165889  0.178312     0.132314   
Certificate         0.122704     1.000000  0.016269  0.120414     0.024708   
Runtime             0.165889     0.016269  1.000000  0.062285     0.243096   
Genre               0.178312     0.120414  0.062285  1.000000     0.028939   
IMDB_Rating         0.132314     0.024708  0.243096  0.028939     1.000000   
Meta_score          0.289446     0.032409  0.027794  0.138065     0.253903   
Director            0.098546     0.036088  0.016764  0.051014     0.011944   
Star1               0.014188     0.040511  0.001729  0.051163     0.019455   
Star2               0.055974     0.004142  0.056766  0.017573     0.029169   
Star3               0.000232     0.026168  0.011570  0.006990     0.016579   
Star4               0.008729     0.034269  0.031742  0.023571     0.023168   
No_of_Votes         0.241624     0.054933  0.173264  0.177041   

In [212]:
high_corr_pairs = []
for col in upper.columns:
    for row in upper.index:
        if upper.loc[row, col] is not np.nan and upper.loc[row, col] > 0.100:
            high_corr_pairs.append([row, col, round(upper.loc[row, col], 2)])

corr_table = pd.DataFrame(high_corr_pairs, columns=['Feature 1', 'Feature 2', 'Correlation'])
print("Highly correlated feature pairs (correlation > 0.7):")
print(corr_table)

Highly correlated feature pairs (correlation > 0.7):
        Feature 1    Feature 2  Correlation
0   Released_Year  Certificate         0.12
1   Released_Year      Runtime         0.17
2   Released_Year        Genre         0.18
3     Certificate        Genre         0.12
4   Released_Year  IMDB_Rating         0.13
5         Runtime  IMDB_Rating         0.24
6   Released_Year   Meta_score         0.29
7           Genre   Meta_score         0.14
8     IMDB_Rating   Meta_score         0.25
9   Released_Year  No_of_Votes         0.24
10        Runtime  No_of_Votes         0.17
11          Genre  No_of_Votes         0.18
12    IMDB_Rating  No_of_Votes         0.49
13  Released_Year        Gross         0.19
14    Certificate        Gross         0.17
15        Runtime        Gross         0.12
16          Genre        Gross         0.30
17    No_of_Votes        Gross         0.55


In [213]:
df_filtered = df.drop(columns=to_drop)
print("\nOriginal shape:", df.shape)
print("Shape after dropping correlated features:", df_filtered.shape)


Original shape: (1000, 13)
Shape after dropping correlated features: (1000, 13)


In [214]:
import plotly.express as px
corr_long = corr_matrix.reset_index().melt(id_vars='index')
corr_long.columns = ['Feature 1', 'Feature 2', 'Correlation']

fig = px.imshow(
    corr_matrix,
    text_auto='.2f',       
    aspect="auto",
    color_continuous_scale='RdBu_r', 
    zmin=-1, zmax=1,
)

fig.update_layout(
    width=900,
    height=800,
    xaxis_title="Features",
    yaxis_title="Features"
)

fig.show()

In [215]:
#Low Vairance
from sklearn.feature_selection import VarianceThreshold

# Set threshold for variance (e.g., 0.01)
threshold = 0.01

# Initialize VarianceThreshold
selector = VarianceThreshold(threshold=threshold)

# Fit on the dataset
selector.fit(df_filtered)

# Features with low variance
low_variance_features = df_filtered.columns[~selector.get_support()]
print("Features with low variance (to drop):", list(low_variance_features))

# Drop low-variance features
df_low_variance_filtered = df_filtered.drop(columns=low_variance_features)
print("\nOriginal shape:", df_filtered.shape)
print("Shape after dropping low-variance features:", df_low_variance_filtered.shape)

Features with low variance (to drop): []

Original shape: (1000, 13)
Shape after dropping low-variance features: (1000, 13)


In [216]:
low_variance_features = df_filtered.columns[~selector.get_support()]
print("Features with low variance (to drop):", list(low_variance_features))

Features with low variance (to drop): []


In [217]:
#Ploty express
numeric_cols = df.select_dtypes(include=[np.number]).columns
variances = df[numeric_cols].var()

threshold = 0.01
low_variance_features = variances[variances < threshold].index.tolist()

var_df = pd.DataFrame({
    'Feature': variances.index,
    'Variance': variances.values,
    'LowVariance': ['Yes' if f in low_variance_features else 'No' for f in variances.index]
})


fig = px.bar(
    var_df,
    x='Feature',
    y='Variance',
    color='LowVariance',
    color_discrete_map={'Yes': 'red', 'No': 'blue'},
    text='Variance',
    title=''
)

fig.update_layout(
    xaxis_tickangle=-45,
    width=1000,
    height=600
)

fig.show()

In [218]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Released_Year  1000 non-null   float64
 1   Certificate    1000 non-null   float64
 2   Runtime        1000 non-null   float64
 3   Genre          1000 non-null   float64
 4   IMDB_Rating    1000 non-null   float64
 5   Meta_score     1000 non-null   float64
 6   Director       1000 non-null   float64
 7   Star1          1000 non-null   float64
 8   Star2          1000 non-null   float64
 9   Star3          1000 non-null   float64
 10  Star4          1000 non-null   float64
 11  No_of_Votes    1000 non-null   float64
 12  Gross          1000 non-null   float64
dtypes: float64(13)
memory usage: 101.7 KB


In [219]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

X = df.drop('IMDB_Rating', axis=1)
y = df['IMDB_Rating']

In [220]:
model = LinearRegression()
rfe = RFE(model, n_features_to_select=10)
rfe.fit(X, y)

selected_features = X.columns[rfe.support_]
print("Selected features:", selected_features)

Selected features: Index(['Released_Year', 'Certificate', 'Runtime', 'Genre', 'Meta_score',
       'Director', 'Star1', 'Star2', 'No_of_Votes', 'Gross'],
      dtype='str')


In [221]:
#For linear models
from sklearn.linear_model import LassoCV
X = df.drop('IMDB_Rating', axis=1)
y = df['IMDB_Rating']
lasso = LassoCV(cv=10, random_state=42).fit(X, y)
importance = np.abs(lasso.coef_)

In [222]:
selected_features = X.columns[importance > 0]

print("Selected features using Lasso (non-zero coefficients):")
print(selected_features.tolist())

Selected features using Lasso (non-zero coefficients):
['Released_Year', 'Certificate', 'Runtime', 'Genre', 'Meta_score', 'Director', 'Star1', 'Star2', 'Star3', 'Star4', 'No_of_Votes', 'Gross']


In [223]:
percentile_threshold = np.percentile(importance, 75)  # top 25% features
top_features = X.columns[importance >= percentile_threshold]

print("\nTop 25% important features based on Lasso coefficients:")
print(top_features.tolist())


Top 25% important features based on Lasso coefficients:
['Runtime', 'No_of_Votes', 'Gross']


In [224]:
feat_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importance
})
feat_df = feat_df.sort_values(by='Importance', ascending=False)

# Mark top 25% most important features
percentile_threshold = np.percentile(importance, 65)
feat_df['Top25%'] = np.where(feat_df['Importance'] >= percentile_threshold, 'Yes', 'No')

# Show table
print("Feature Importance Table:")
print(feat_df)

Feature Importance Table:
          Feature  Importance Top25%
10    No_of_Votes    0.778481    Yes
11          Gross    0.399001    Yes
2         Runtime    0.321293    Yes
4      Meta_score    0.201883    Yes
0   Released_Year    0.161001     No
1     Certificate    0.049405     No
7           Star2    0.031124     No
6           Star1    0.025025     No
3           Genre    0.017432     No
9           Star4    0.010153     No
5        Director    0.010144     No
8           Star3    0.003654     No


In [225]:
fig = px.bar(
    feat_df,
    x='Feature',
    y='Importance',
    color='Top25%',
    color_discrete_map={'Yes': 'red', 'No': 'blue'},
    text='Importance',
    title='Lasso Feature Importance'
)

fig.update_layout(
    xaxis_tickangle=-45,
    width=1000,
    height=600
)


In [226]:
# Tree based(Random Forest)
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(n_estimators=500, random_state=42)
rf.fit(X, y)
importances = rf.feature_importances_

# Create a DataFrame for feature importance
feat_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importances
})
feat_df = feat_df.sort_values(by='Importance', ascending=False)

# Mark top 25% most important features
percentile_threshold = np.percentile(importances, 50)
feat_df['Top25%'] = np.where(feat_df['Importance'] >= percentile_threshold, 'Yes', 'No')

# Show table
print("Random Forest Feature Importance Table:")
print(feat_df)

Random Forest Feature Importance Table:
          Feature  Importance Top25%
10    No_of_Votes    0.397438    Yes
4      Meta_score    0.129088    Yes
0   Released_Year    0.094411    Yes
11          Gross    0.065937    Yes
2         Runtime    0.064870    Yes
7           Star2    0.044252    Yes
6           Star1    0.043280     No
5        Director    0.038390     No
8           Star3    0.037151     No
3           Genre    0.035332     No
9           Star4    0.034497     No
1     Certificate    0.015355     No


In [227]:
fig = px.bar(
    feat_df,
    x='Feature',
    y='Importance',
    color='Top25%',
    color_discrete_map={'Yes': 'red', 'No': 'blue'},
    text='Importance',
    title='Random Forest Feature Importance'
)

fig.update_layout(
    xaxis_tickangle=-45,
    width=1000,
    height=600
)

fig.show()


# Algoritms 
# Boosting(XGBoost) larsiz !!! qilingani 

In [228]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Released_Year  1000 non-null   float64
 1   Certificate    1000 non-null   float64
 2   Runtime        1000 non-null   float64
 3   Genre          1000 non-null   float64
 4   IMDB_Rating    1000 non-null   float64
 5   Meta_score     1000 non-null   float64
 6   Director       1000 non-null   float64
 7   Star1          1000 non-null   float64
 8   Star2          1000 non-null   float64
 9   Star3          1000 non-null   float64
 10  Star4          1000 non-null   float64
 11  No_of_Votes    1000 non-null   float64
 12  Gross          1000 non-null   float64
dtypes: float64(13)
memory usage: 101.7 KB


In [229]:
import pandas as pd
from tabulate import tabulate
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error


X = df.select_dtypes(include=['int64', 'float64']).drop(['Sale_ID'], axis=1, errors='ignore')
y = df['IMDB_Rating']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "SVM (SVR)": SVR(kernel='rbf')
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    
    score = model.score(X_test, y_test)
    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    
    results.append([name, f"{score:.3f}", f"{mae:.3f}", f"{mse:.3f}"])

headers = ["Model", "R2 Score", "MAE", "MSE"]
print(tabulate(results, headers=headers, tablefmt="grid"))

+-------------------+------------+-------+-------+
| Model             |   R2 Score |   MAE |   MSE |
+===================+============+=======+=======+
| Linear Regression |      1     | 0     | 0     |
+-------------------+------------+-------+-------+
| Decision Tree     |      1     | 0     | 0     |
+-------------------+------------+-------+-------+
| Random Forest     |      1     | 0     | 0     |
+-------------------+------------+-------+-------+
| SVM (SVR)         |      0.841 | 0.046 | 0.004 |
+-------------------+------------+-------+-------+


# Boosting(XGBoost) bilan qilingan Algorithms !!!

In [231]:
import pandas as pd
import xgboost as xgb
from tabulate import tabulate
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error

X = df.select_dtypes(include=['int64', 'float64']).drop(['Sale_ID'], axis=1, errors='ignore')
y = df['IMDB_Rating']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "SVM (SVR)": SVR(kernel='rbf'),
    "XGBoost": xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    
    score = model.score(X_test, y_test)
    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    
    results.append([name, f"{score:.4f}", f"{mae:.4f}", f"{mse:.4f}"])

headers = ["Model", "R2 Score", "MAE", "MSE"]
print(tabulate(results, headers=headers, tablefmt="grid"))

+-------------------+------------+--------+--------+
| Model             |   R2 Score |    MAE |    MSE |
+===================+============+========+========+
| Linear Regression |     1      | 0      | 0      |
+-------------------+------------+--------+--------+
| Decision Tree     |     1      | 0      | 0      |
+-------------------+------------+--------+--------+
| Random Forest     |     1      | 0.0001 | 0      |
+-------------------+------------+--------+--------+
| SVM (SVR)         |     0.8408 | 0.0457 | 0.0035 |
+-------------------+------------+--------+--------+
| XGBoost           |     1      | 0      | 0      |
+-------------------+------------+--------+--------+
